# Tech Challenge Fase 3 - Analises no AWS Glue Notebook

**Pesquisa State of Data Brasil | 3 ultimas edicoes | Data Lake Bronze / Silver / Gold**

| Edicao no Kaggle | `ano` no projeto | Respondentes |
| --- | --- | --- |
| State of Data Brazil 2023-2024 | 2023 | 5.293 |
| State of Data Brazil 2024-2025 | 2024 | 5.217 |
| State of Data Brazil 2025-2026 | 2025 | 3.495 |

Este notebook executa as consultas analiticas sobre as camadas Silver e Gold usando **Spark SQL** e gera os graficos do material executivo.

Pre-requisitos: `job_01_bronze_para_silver.py` e `job_02_silver_para_gold.py` executados com sucesso.

> Ajuste a variavel `BUCKET` na celula de configuracao antes de executar.

## Decisoes metodologicas que governam este notebook

Tres cuidados foram necessarios porque os dados brutos induzem a erro:

1. **Remuneracao: use `salario_mediano_interpolado`.** A pesquisa coleta renda em faixas. A mediana do ponto medio da faixa e cega: ela devolve o centro da faixa mediana e fazia *todas* as regioes exibirem exatamente R$ 10.000. A metrica correta para dados agrupados e a mediana interpolada.

2. **Adocao de tecnologia: o denominador e `base_resposta`**, nao o total de respondentes da edicao. A cobertura das questoes de tecnologia cai de 71% (2023) para 60% (2025); usar o total faria toda tecnologia parecer perder ~11 pontos por artefato de base.

3. **Nem toda serie e comparavel entre anos.** `linguagem_uso` nao existe em 2025, e `linguagem_preferida` passou de escolha unica para multipla na mesma edicao (o SQL "saltaria" de 1,4% para 50,5% por mudanca de questionario). A coluna `itens_por_respondente` permite detectar isso.

## 1. Configuracao da sessao interativa do Glue

As magics abaixo valem apenas no Glue Notebook. O `idle_timeout` curto e o numero baixo de workers ajudam a nao estourar o orcamento do AWS Academy Lab.

In [ ]:
%idle_timeout 30
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

BUCKET = "SEU-BUCKET"  # <<< ajuste aqui
SILVER = f"s3://{BUCKET}/silver"
GOLD = f"s3://{BUCKET}/gold"

print(spark.version)

## 2. Carga das camadas e registro das views

In [ ]:
respondentes = spark.read.parquet(f"{SILVER}/respondentes/")
tecnologias = spark.read.parquet(f"{SILVER}/tecnologias/")

respondentes.createOrReplaceTempView("respondentes")
tecnologias.createOrReplaceTempView("tecnologias")

TABELAS_GOLD = [
    "perfil_mercado", "remuneracao", "diversidade",
    "tecnologias", "adocao_ia", "impacto_ia", "modelo_trabalho",
]
for nome in TABELAS_GOLD:
    spark.read.parquet(f"{GOLD}/{nome}/").createOrReplaceTempView(f"gold_{nome}")

print(f"respondentes: {respondentes.count()} linhas")
respondentes.printSchema()

## 3. Sanidade dos dados

Antes de qualquer analise: volume por edicao e cobertura dos campos criticos.

In [ ]:
spark.sql("""
    SELECT ano,
           COUNT(*)                                                    AS respondentes,
           COUNT(salario_mensal_estimado)                              AS com_salario,
           ROUND(100.0 * COUNT(salario_mensal_estimado) / COUNT(*), 1) AS pct_com_salario,
           COUNT(usa_ia)                                               AS com_resposta_ia,
           SUM(CASE WHEN senioridade = 'Nao informado' THEN 1 ELSE 0 END) AS sem_senioridade
    FROM respondentes
    GROUP BY ano
    ORDER BY ano
""").show()

## 4. Como esta estruturado o mercado brasileiro de dados?

A senioridade dos gestores e reconstruida no Job 1: a pesquisa nao pergunta o nivel a quem atua como gestor, entao 2.668 respondentes cairiam em "Nao informado" sem esse tratamento.

In [ ]:
spark.sql("""
    SELECT senioridade,
           SUM(CASE WHEN ano = 2023 THEN respondentes END) AS resp_2023,
           SUM(CASE WHEN ano = 2024 THEN respondentes END) AS resp_2024,
           SUM(CASE WHEN ano = 2025 THEN respondentes END) AS resp_2025
    FROM gold_perfil_mercado
    GROUP BY senioridade
    ORDER BY resp_2025 DESC
""").show()

In [ ]:
spark.sql("""
    SELECT cargo, SUM(respondentes) AS respondentes
    FROM gold_perfil_mercado
    WHERE ano = (SELECT MAX(ano) FROM gold_perfil_mercado) AND cargo IS NOT NULL
    GROUP BY cargo
    ORDER BY respondentes DESC
    LIMIT 15
""").show(truncate=False)

## 5. Quais perfis sao mais valorizados pelo mercado?

`gold_remuneracao` esta em formato longo: a coluna `recorte` define quais dimensoes estao em `dimensao_1` e `dimensao_2`. Cruzar todas as dimensoes de uma vez pulverizaria a amostra, por isso cada recorte e agregado separadamente. O filtro `amostra_suficiente` descarta grupos com menos de 30 respondentes.

In [ ]:
spark.sql("""
    SELECT dimensao_1 AS cargo, respondentes,
           salario_mediano_interpolado AS salario, p25, p75
    FROM gold_remuneracao
    WHERE recorte = 'cargo' AND amostra_suficiente AND dimensao_1 IS NOT NULL
      AND ano = (SELECT MAX(ano) FROM gold_remuneracao)
    ORDER BY salario DESC
""").show(truncate=False)

In [ ]:
# Progressao salarial por senioridade nas 3 edicoes
spark.sql("""
    SELECT dimensao_1 AS senioridade,
           MAX(CASE WHEN ano = 2023 THEN salario_mediano_interpolado END) AS salario_2023,
           MAX(CASE WHEN ano = 2024 THEN salario_mediano_interpolado END) AS salario_2024,
           MAX(CASE WHEN ano = 2025 THEN salario_mediano_interpolado END) AS salario_2025
    FROM gold_remuneracao
    WHERE recorte = 'senioridade' AND amostra_suficiente
    GROUP BY dimensao_1
    ORDER BY salario_2025
""").show()

In [ ]:
# Retorno da experiencia e da formacao: insumo da recomendacao de capacitacao
spark.sql("""
    SELECT recorte, dimensao_1, respondentes,
           salario_mediano_interpolado AS salario
    FROM gold_remuneracao
    WHERE recorte IN ('tempo_experiencia_dados', 'nivel_ensino')
      AND amostra_suficiente AND dimensao_1 IS NOT NULL
      AND ano = (SELECT MAX(ano) FROM gold_remuneracao)
    ORDER BY recorte, salario DESC
""").show(30, truncate=False)

## 6. Diversidade de genero nas carreiras de dados

A participacao feminina por senioridade revela o afunilamento da carreira, que e mais informativo do que o total agregado.

In [ ]:
spark.sql("""
    SELECT senioridade,
           SUM(CASE WHEN genero = 'Feminino'  THEN respondentes ELSE 0 END) AS mulheres,
           SUM(CASE WHEN genero = 'Masculino' THEN respondentes ELSE 0 END) AS homens,
           ROUND(100.0 * SUM(CASE WHEN genero = 'Feminino' THEN respondentes ELSE 0 END)
                 / NULLIF(SUM(CASE WHEN genero IN ('Feminino','Masculino') THEN respondentes ELSE 0 END), 0), 1)
               AS pct_feminino
    FROM gold_diversidade
    GROUP BY senioridade
    ORDER BY pct_feminino DESC
""").show()

In [ ]:
spark.sql("""
    WITH base AS (
        SELECT senioridade,
               MAX(CASE WHEN genero = 'Masculino' THEN salario_mediano_interpolado END) AS salario_masc,
               MAX(CASE WHEN genero = 'Feminino'  THEN salario_mediano_interpolado END) AS salario_fem
        FROM gold_diversidade
        WHERE ano = (SELECT MAX(ano) FROM gold_diversidade)
        GROUP BY senioridade
    )
    SELECT senioridade, salario_masc, salario_fem,
           ROUND(100.0 * (salario_fem - salario_masc) / salario_masc, 1) AS diferenca_pct
    FROM base
    WHERE salario_masc IS NOT NULL AND salario_fem IS NOT NULL
    ORDER BY diferenca_pct
""").show()

## 7. Tecnologias com maior adocao e sua evolucao

Antes de comparar anos, confira `itens_por_respondente`: se ele mudou entre edicoes, a questao mudou de formato e a serie nao e comparavel.

In [ ]:
# Diagnostico de comparabilidade das categorias
spark.sql("""
    SELECT categoria,
           MAX(CASE WHEN ano = 2023 THEN itens_por_respondente END) AS itens_2023,
           MAX(CASE WHEN ano = 2024 THEN itens_por_respondente END) AS itens_2024,
           MAX(CASE WHEN ano = 2025 THEN itens_por_respondente END) AS itens_2025,
           MAX(CASE WHEN ano = 2025 THEN base_resposta END)         AS base_2025
    FROM gold_tecnologias
    GROUP BY categoria
    ORDER BY categoria
""").show()

In [ ]:
spark.sql("""
    SELECT categoria, tecnologia, respondentes, base_resposta, taxa_adocao, posicao
    FROM gold_tecnologias
    WHERE ano = (SELECT MAX(ano) FROM gold_tecnologias) AND posicao <= 10
    ORDER BY categoria, posicao
""").show(60, truncate=False)

In [ ]:
# Evolucao apenas nas categorias com formato estavel nas 3 edicoes
spark.sql("""
    WITH comparavel AS (
        SELECT * FROM gold_tecnologias
        WHERE categoria IN ('banco_dados', 'cloud_uso', 'cloud_preferida', 'ferramenta_bi')
    ),
    limites AS (SELECT MIN(ano) AS ano_ini, MAX(ano) AS ano_fim FROM comparavel),
    comparativo AS (
        SELECT c.categoria, c.tecnologia,
               MAX(CASE WHEN c.ano = l.ano_ini THEN c.taxa_adocao END) AS adocao_inicial,
               MAX(CASE WHEN c.ano = l.ano_fim THEN c.taxa_adocao END) AS adocao_final
        FROM comparavel c CROSS JOIN limites l
        GROUP BY c.categoria, c.tecnologia
    )
    SELECT categoria, tecnologia, adocao_inicial, adocao_final,
           ROUND(adocao_final - adocao_inicial, 2) AS variacao_pp
    FROM comparativo
    WHERE adocao_inicial IS NOT NULL AND adocao_final IS NOT NULL
      AND (adocao_inicial >= 5 OR adocao_final >= 5)
    ORDER BY variacao_pp DESC
""").show(40, truncate=False)

## 8. Adocao de Inteligencia Artificial e seu impacto

O filtro `base_resposta > 0` exclui grupos sem resposta a questao. A comparacao salarial entre quem usa e quem nao usa IA e uma **associacao**, nao uma relacao causal.

In [ ]:
spark.sql("""
    SELECT ano, senioridade, base_resposta, taxa_adocao_ia
    FROM gold_adocao_ia
    WHERE base_resposta > 0
    ORDER BY ano, taxa_adocao_ia DESC
""").show(30)

In [ ]:
spark.sql("""
    SELECT ano, senioridade,
           MAX(CASE WHEN grupo = 'Usa IA'     THEN salario_mediano_interpolado END) AS salario_usa_ia,
           MAX(CASE WHEN grupo = 'Nao usa IA' THEN salario_mediano_interpolado END) AS salario_nao_usa_ia,
           MAX(CASE WHEN grupo = 'Usa IA'     THEN respondentes END)                AS n_usa_ia,
           MAX(CASE WHEN grupo = 'Nao usa IA' THEN respondentes END)                AS n_nao_usa_ia
    FROM gold_impacto_ia
    WHERE amostra_suficiente
    GROUP BY ano, senioridade
    ORDER BY ano, senioridade
""").show(30)

In [ ]:
# Perfil do uso de IA: gratuita, paga ou assistente de codigo
spark.sql("""
    SELECT ano, tipo_uso_ia, COUNT(*) AS respondentes,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY ano), 2) AS pct_do_ano
    FROM respondentes
    WHERE tipo_uso_ia IS NOT NULL
    GROUP BY ano, tipo_uso_ia
    ORDER BY ano, respondentes DESC
""").show(30, truncate=False)

## 9. Diferencas por regiao e modelo de trabalho

In [ ]:
spark.sql("""
    SELECT ano, dimensao_1 AS regiao, respondentes,
           ROUND(100.0 * respondentes / SUM(respondentes) OVER (PARTITION BY ano), 2) AS pct_do_ano,
           salario_mediano_interpolado AS salario
    FROM gold_remuneracao
    WHERE recorte = 'regiao' AND dimensao_1 <> 'Nao informado'
    ORDER BY ano, respondentes DESC
""").show(30)

In [ ]:
spark.sql("""
    SELECT dimensao_1 AS forma_trabalho,
           MAX(CASE WHEN ano = 2025 THEN respondentes END)                AS n_2025,
           MAX(CASE WHEN ano = 2023 THEN salario_mediano_interpolado END) AS salario_2023,
           MAX(CASE WHEN ano = 2025 THEN salario_mediano_interpolado END) AS salario_2025
    FROM gold_remuneracao
    WHERE recorte = 'forma_trabalho' AND dimensao_1 <> 'Nao informado' AND amostra_suficiente
    GROUP BY dimensao_1
    ORDER BY salario_2025 DESC
""").show()

## 10. Graficos para o material executivo

As agregacoes da Gold sao pequenas, portanto podem ser trazidas para o driver com `toPandas()` sem risco de estourar memoria.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

PALETA = ["#1f4e79", "#2e75b6", "#9dc3e6", "#f4b183", "#c55a11"]
ORDEM = ["Junior", "Pleno", "Senior", "Especialista", "Gestao"]

pdf = (
    spark.sql("""
        SELECT ano, dimensao_1 AS senioridade, salario_mediano_interpolado AS salario
        FROM gold_remuneracao
        WHERE recorte = 'senioridade' AND amostra_suficiente
    """)
    .toPandas()
    .pivot(index="ano", columns="senioridade", values="salario")
)

colunas = [c for c in ORDEM if c in pdf.columns]
ax = pdf[colunas].plot(kind="bar", figsize=(9, 5), color=PALETA)
ax.set_title("Remuneracao mensal por senioridade")
ax.set_xlabel("Ano da pesquisa")
ax.set_ylabel("R$ / mes (mediana interpolada)")
plt.tight_layout()
plt.show()

In [ ]:
pdf_ia = (
    spark.sql("""
        SELECT senioridade, ano, taxa_adocao_ia
        FROM gold_adocao_ia
        WHERE base_resposta > 0
    """)
    .toPandas()
    .pivot(index="senioridade", columns="ano", values="taxa_adocao_ia")
)

linhas = [i for i in ORDEM if i in pdf_ia.index]
ax = pdf_ia.loc[linhas].plot(kind="bar", figsize=(9, 5), color=PALETA)
ax.set_title("Adocao de IA generativa por senioridade")
ax.set_xlabel("Senioridade")
ax.set_ylabel("% que usa IA no trabalho")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
pdf_tec = (
    spark.sql("""
        SELECT tecnologia, taxa_adocao
        FROM gold_tecnologias
        WHERE categoria = 'cloud_uso'
          AND ano = (SELECT MAX(ano) FROM gold_tecnologias)
          AND posicao <= 10
        ORDER BY taxa_adocao
    """)
    .toPandas()
    .set_index("tecnologia")
)

ax = pdf_tec.plot(kind="barh", figsize=(9, 5), color=PALETA[1], legend=False)
ax.set_title("Cloud utilizada no dia a dia - taxa de adocao")
ax.set_xlabel("% de quem respondeu a questao")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 11. Exportacao dos resultados para a apresentacao

In [ ]:
for nome in TABELAS_GOLD:
    (
        spark.table(f"gold_{nome}")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", "true")
        .csv(f"s3://{BUCKET}/exportacao/{nome}/")
    )
    print(f"exportado {nome}")

## 12. Limitacoes metodologicas

Pontos que devem constar no material executivo:

- **Amostra voluntaria (auto-selecao)**: os respondentes sao membros da comunidade Data Hackers, com vies para profissionais engajados e digitalmente ativos. Nao representa o mercado brasileiro como um todo.
- **Renda em faixas**: a pesquisa coleta faixa, nao valor exato. Usamos mediana interpolada; faixas abertas no topo ("Acima de R$ 40.001") usam o limite inferior, o que **subestima** o topo da distribuicao.
- **Amostra decrescente**: 5.293 (2023), 5.217 (2024) e 3.495 (2025) respondentes. A ultima edicao tem cerca de um terco menos respostas, o que amplia a incerteza dos recortes menores.
- **Comparabilidade entre edicoes**: o questionario muda. `linguagem_uso` desapareceu em 2025 e `linguagem_preferida` virou multipla escolha; a cobertura das questoes de tecnologia caiu de 71% para 60%.
- **Concentracao geografica**: o Sudeste responde por cerca de 61% a 64% da amostra, o que limita conclusoes sobre Norte e Centro-Oeste (a regiao Norte tem apenas 36 respondentes com salario em 2025).
- **Senioridade dos gestores reconstruida**: derivada da questao "atua como gestor", pois a pesquisa nao pergunta o nivel a esse grupo.
- **Associacao nao e causalidade**: diferencas salariais entre grupos nao controlam simultaneamente senioridade, setor, porte de empresa e regiao.